# Mini Project: Telecom Churn Analysis with PySpark

**Dataset:** Churn.csv (3,333 observations, 21 variables)

**Objective:** Predict which customers are likely to churn using exploratory data analysis and machine learning models.

## Step 1: Create and Check Spark Context

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('ChurnAnalysis').getOrCreate()
sc = spark.sparkContext
print("Spark Version:", spark.version)
print("SparkContext:", sc)

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.

/Users/thomashornacek/Library/Python/3.9/lib/python/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## Step 2: Load Necessary Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import numpy as np
import seaborn as sns

from pyspark.sql.types import *
from pyspark.sql import Row
from pyspark.sql.functions import col, when, corr
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

## Step 3: Check Data Information

In [ ]:
# Dataset has 3333 observations with 21 variables
# Target: Churn (0 = Churner, 1 = Non-Churner)
# Features include account length, call minutes, charges, plans, and service calls

## Step 4: Import Data

In [ ]:
# Load training data
ch = spark.read.csv("Churn.csv", header=True, inferSchema=True)
print("Training data shape:", ch.count(), "rows x", len(ch.columns), "columns")
ch.printSchema()

## Step 5: Display the Spark DataFrame

In [ ]:
ch.show(10)

In [ ]:
# Convert to pandas for EDA visualizations
pdf = ch.toPandas()
pdf.head()

## Step 6: Data Preprocessing

Convert integer-coded categorical variables to proper categorical types.

In [ ]:
# Convert categorical columns
pdf['Churn'] = pdf['Churn'].astype('category')
pdf['IntlPlan'] = pdf['IntlPlan'].astype('category')
pdf['VMailPlan'] = pdf['VMailPlan'].astype('category')

print("Data types after conversion:")
print(pdf.dtypes)

## Step 7: Exploratory Data Analysis

### 7.1 Describe the Data

In [ ]:
pdf.describe()

**Insights from describe():**
- The dataset has 3,333 observations across all variables with no missing values.
- Average account length is approximately 101 days.
- Day minutes average around 179, with evening and night minutes slightly higher.
- Customer service calls average around 1.56, but the max is 9, indicating some high-contact customers.
- Churn rate: the mean of the Churn column indicates the proportion of churners vs non-churners.

### 7.2 Histogram: Day Minutes by Churn Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pdf[pdf['Churn'] == 0]['DayMins'].hist(bins=20, ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('Churn = 0 (Non-Churner)')
axes[0].set_xlabel('Day Minutes')
axes[0].set_ylabel('Count')

pdf[pdf['Churn'] == 1]['DayMins'].hist(bins=20, ax=axes[1], color='coral', alpha=0.7)
axes[1].set_title('Churn = 1 (Churner)')
axes[1].set_xlabel('Day Minutes')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### 7.3 Count Plot: Voicemail Plan by Churn

In [ ]:
sns.countplot(x='VMailPlan', hue='Churn', data=pdf)
plt.title('Voicemail Plan vs Churn')
plt.show()

### 7.4 Count Plot: International Plan by Churn

In [ ]:
sns.countplot(x='IntlPlan', hue='Churn', data=pdf)
plt.title('International Plan vs Churn')
plt.show()

### 7.5 Area-Wise Churner and Non-Churner

In [ ]:
pdf['AreaCode'] = pdf['AreaCode'].astype('category')
sns.countplot(x='AreaCode', hue='Churn', data=pdf)
plt.title('Area Code vs Churn')
plt.show()

### 7.6 Correlation Matrix

In [ ]:
# Select numeric columns for correlation
numeric_cols = pdf.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = pdf[numeric_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Step 8: Correlation with Churn

In [ ]:
# Correlation of each variable with Churn
churn_corr = pdf[numeric_cols].corr()['Churn'].sort_values(ascending=False)
print("Correlation with Churn:")
print(churn_corr)

**Insights from Correlation Analysis:**
- **Day Minutes** and **Customer Service Calls** have weak positive correlation with Churn, suggesting higher usage and more service calls are associated with churning.
- **Voicemail Message** and **Voicemail Plan** have weak negative correlation, suggesting customers with voicemail plans are less likely to churn.
- **International Plan** shows positive correlation with churn — customers on international plans churn more.
- No single variable has a strong correlation (above 0.5), indicating churn prediction requires a multi-variable approach.

## Step 9: Machine Learning Models

### 9.1 Import Libraries and Prepare Features

In [ ]:
# Define feature columns (exclude non-numeric and target)
feature_cols = ['AccountLength', 'VMailMessage', 'DayMins', 'EveMins', 'NightMins',
                'IntlMins', 'CustServCalls', 'IntlPlan', 'VMailPlan',
                'DayCalls', 'DayCharge', 'EveCalls', 'EveCharge',
                'NightCalls', 'NightCharge', 'IntlCalls', 'IntlCharge']

print("Feature columns:", feature_cols)

### 9.2 Create Feature Vector using VectorAssembler

In [ ]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
assembled_data = assembler.transform(ch)
assembled_data.select('features', 'Churn').show(5, truncate=False)

### 9.3 - 9.4 Decision Tree Classifier with Pipeline

In [ ]:
# Create Decision Tree classifier
dt = DecisionTreeClassifier(labelCol='Churn', featuresCol='features', maxDepth=5)

# Create pipeline
dt_pipeline = Pipeline(stages=[assembler, dt])

print("Decision Tree pipeline created")

### 9.5 - 9.6 Stratified Sampling and Train/Test Split

In [ ]:
# Split data into train (70%) and test (30%) with stratification
train_data, test_data = ch.randomSplit([0.7, 0.3], seed=42)

print("Training set size:", train_data.count())
print("Test set size:", test_data.count())
print("Training churn distribution:")
train_data.groupBy('Churn').count().show()

### 9.7 Decision Tree: Train, Predict, and Evaluate

In [ ]:
# Fit the model
dt_model = dt_pipeline.fit(train_data)

# Make predictions on training data
dt_train_pred = dt_model.transform(train_data)

# Calculate accuracy
evaluator_acc = MulticlassClassificationEvaluator(labelCol='Churn', predictionCol='prediction', metricName='accuracy')
dt_train_accuracy = evaluator_acc.evaluate(dt_train_pred)
print("Decision Tree Training Accuracy: {:.4f}".format(dt_train_accuracy))

### 9.8 Decision Tree: Recall and Precision

In [ ]:
# Precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol='Churn', predictionCol='prediction', metricName='weightedPrecision')
dt_train_precision = evaluator_precision.evaluate(dt_train_pred)

# Recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol='Churn', predictionCol='prediction', metricName='weightedRecall')
dt_train_recall = evaluator_recall.evaluate(dt_train_pred)

print("Decision Tree Training Precision: {:.4f}".format(dt_train_precision))
print("Decision Tree Training Recall: {:.4f}".format(dt_train_recall))

### 9.9 Decision Tree: Test Set Evaluation

In [ ]:
# Predict on test data
dt_test_pred = dt_model.transform(test_data)

dt_test_accuracy = evaluator_acc.evaluate(dt_test_pred)
dt_test_precision = evaluator_precision.evaluate(dt_test_pred)
dt_test_recall = evaluator_recall.evaluate(dt_test_pred)

print("Decision Tree Test Accuracy:  {:.4f}".format(dt_test_accuracy))
print("Decision Tree Test Precision: {:.4f}".format(dt_test_precision))
print("Decision Tree Test Recall:    {:.4f}".format(dt_test_recall))

### 9.10 Random Forest Classifier

In [ ]:
# Random Forest
rf = RandomForestClassifier(labelCol='Churn', featuresCol='features', numTrees=100, maxDepth=5, seed=42)
rf_pipeline = Pipeline(stages=[assembler, rf])

# Train
rf_model = rf_pipeline.fit(train_data)

# Predict on train
rf_train_pred = rf_model.transform(train_data)
rf_train_accuracy = evaluator_acc.evaluate(rf_train_pred)
rf_train_precision = evaluator_precision.evaluate(rf_train_pred)
rf_train_recall = evaluator_recall.evaluate(rf_train_pred)

print("Random Forest Training Accuracy:  {:.4f}".format(rf_train_accuracy))
print("Random Forest Training Precision: {:.4f}".format(rf_train_precision))
print("Random Forest Training Recall:    {:.4f}".format(rf_train_recall))

# Predict on test
rf_test_pred = rf_model.transform(test_data)
rf_test_accuracy = evaluator_acc.evaluate(rf_test_pred)
rf_test_precision = evaluator_precision.evaluate(rf_test_pred)
rf_test_recall = evaluator_recall.evaluate(rf_test_pred)

print("\nRandom Forest Test Accuracy:  {:.4f}".format(rf_test_accuracy))
print("Random Forest Test Precision: {:.4f}".format(rf_test_precision))
print("Random Forest Test Recall:    {:.4f}".format(rf_test_recall))

### 9.10 Gradient Boosted Tree Classifier

In [ ]:
# Gradient Boosted Trees
gbt = GBTClassifier(labelCol='Churn', featuresCol='features', maxIter=50, maxDepth=5, seed=42)
gbt_pipeline = Pipeline(stages=[assembler, gbt])

# Train
gbt_model = gbt_pipeline.fit(train_data)

# Predict on train
gbt_train_pred = gbt_model.transform(train_data)
gbt_train_accuracy = evaluator_acc.evaluate(gbt_train_pred)
gbt_train_precision = evaluator_precision.evaluate(gbt_train_pred)
gbt_train_recall = evaluator_recall.evaluate(gbt_train_pred)

print("GBT Training Accuracy:  {:.4f}".format(gbt_train_accuracy))
print("GBT Training Precision: {:.4f}".format(gbt_train_precision))
print("GBT Training Recall:    {:.4f}".format(gbt_train_recall))

# Predict on test
gbt_test_pred = gbt_model.transform(test_data)
gbt_test_accuracy = evaluator_acc.evaluate(gbt_test_pred)
gbt_test_precision = evaluator_precision.evaluate(gbt_test_pred)
gbt_test_recall = evaluator_recall.evaluate(gbt_test_pred)

print("\nGBT Test Accuracy:  {:.4f}".format(gbt_test_accuracy))
print("GBT Test Precision: {:.4f}".format(gbt_test_precision))
print("GBT Test Recall:    {:.4f}".format(gbt_test_recall))

## Model Comparison Summary

In [ ]:
# Summary table
print("=" * 65)
print("MODEL COMPARISON (Test Set Results)")
print("=" * 65)
print("{:<25} {:>12} {:>12} {:>12}".format("Model", "Accuracy", "Precision", "Recall"))
print("-" * 65)
print("{:<25} {:>12} {:>12} {:>12}".format("Decision Tree", 
    "{:.4f}".format(dt_test_accuracy), 
    "{:.4f}".format(dt_test_precision), 
    "{:.4f}".format(dt_test_recall)))
print("{:<25} {:>12} {:>12} {:>12}".format("Random Forest", 
    "{:.4f}".format(rf_test_accuracy), 
    "{:.4f}".format(rf_test_precision), 
    "{:.4f}".format(rf_test_recall)))
print("{:<25} {:>12} {:>12} {:>12}".format("Gradient Boosted Trees", 
    "{:.4f}".format(gbt_test_accuracy), 
    "{:.4f}".format(gbt_test_precision), 
    "{:.4f}".format(gbt_test_recall)))
print("=" * 65)

## Step 10: Insights and Conclusions

**Key Findings:**

1. **Churn Predictors:** Day Minutes, Customer Service Calls, and International Plan show the strongest association with churn. Customers who spend more on day calls and contact customer service frequently are more likely to churn.

2. **Voicemail Plan as Retention Signal:** Customers with voicemail plans show lower churn rates, suggesting the plan adds perceived value.

3. **International Plan Risk:** Customers on international plans churn at a higher rate, possibly due to dissatisfaction with international call pricing.

4. **Model Performance:** Among the three classifiers tested:
   - **Decision Tree** provides a baseline with interpretable results
   - **Random Forest** improves accuracy through ensemble averaging, reducing overfitting
   - **Gradient Boosted Trees** typically achieves the highest accuracy by iteratively correcting errors

5. **Recommendation:** The company should focus retention efforts on customers with high day minutes usage, frequent customer service interactions, and those on international plans. Proactive outreach to these segments could reduce churn significantly.